# Banking Campaign Analytics preprocessing

## Business Objective: Identify and analyze the behavioral, demographic, campaign, engagement, risk, and economic factors influencing customer subscription outcomes in order to optimize banking campaign performance, customer targeting, and engagement strategy.


## STEP 1 Importing the Raw Banking Dataset


In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv(
                "bank-additional-full.csv",
                sep=';'
                )
df.head()

## STEP 2 Structural Data Audit

### 2A. Check the dataset shape



In [ ]:
df.shape

### 2B. View all column name



In [ ]:
df.columns

### 2C. Dataset Information summary


In [ ]:
df.info()

### 2D. Preview Dataset


In [ ]:
df.head()

### 2E. Generate Statistical Summary


In [ ]:
df.describe()

### 2F. Check Categorical Summary


In [ ]:
df.describe(include='object')

## STEP 3 Business Column Mapping and Analytical Classification

### 3A. View all columns clearly


In [ ]:
list(df.columns)

### 3B. Business Column Classification

To facilitate a structured analysis, the dataset variables have been organized into five major **Analytical Domains**. This classification allows us to investigate the drivers of conversion from both a micro (customer-level) and macro (market-level) perspective.

---

#### 👤 Customer Demographics
*Variables related to identity and socio-economic profiling.*
* **Fields:** `age`, `job`, `marital`, `education`
* **Analytical Purpose:** Used for **Customer Segmentation** and identifying high-propensity demographic clusters.

#### 💰 Financial Relationship Features
*Variables describing the customer’s existing financial footprint with the bank.*
* **Fields:** `credit_default_status`, `housing_loan_status`, `personal_loan_status`
* **Analytical Purpose:** Used for **Risk Profiling** and determining if existing debt affects the willingness to open a term deposit.

#### 📞 Campaign Interaction Features
*Variables capturing the "mechanics" of the current marketing effort.*
* **Fields:** `contact_method`, `campaign_month`, `campaign_day_of_week`, `call_duration_seconds`, `current_campaign_contacts`
* **Analytical Purpose:** Investigating **Campaign Effectiveness** and identifying the "sweet spot" for contact frequency to avoid fatigue.

#### ⏳ Historical Engagement Features
*Variables describing the long-term relationship and past behaviors.*
* **Fields:** `days_since_previous_contact`, `previous_campaign_contacts`, `previous_campaign_outcome`
* **Analytical Purpose:** Measuring **Relationship Continuity** and the "warmth" of the lead based on prior success.

#### 🌍 Economic Context Features
*External macroeconomic indicators at the time of the call.*
* **Fields:** `employment_variation_rate`, `consumer_price_index`, `consumer_confidence_index`, `market_interest_rate`, `employment_level`
* **Analytical Purpose:** Evaluating how **External Market Conditions** (like inflation or interest rates) suppress or encourage investment.

---

### 🎯 Target Variable (Primary KPI)
* **Variable:** `subscribed`
* **Definition:** A binary indicator representing whether the customer accepted the term deposit offer. This serves as the primary outcome variable for evaluating customer conversion behavior and campaign effectiveness.

### 3C. Identify Columns type

In [ ]:
# Identify numerical columns 
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns
# Identify categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns

# Display numerical columns
print("Numerical Columns:\n")
print(numerical_columns)

# Display categorical columns
print("\nCategorical Columns:\n")
print(categorical_columns)



## STEP 4 Data Quality Audit

### 4A. Check Missing values



In [ ]:
df.isnull().sum()

### 4B. Check for duplicate record

In [ ]:
df.duplicated().sum()

### 4C. Display unique values for categorical columns

In [ ]:
for column in categorical_columns:
    print(f"\n Unique Values in {column}:\n")
    print(df[column].unique())

### 4D. Check the count of 'unknown' values in the categorical columns

In [ ]:
for column in categorical_columns:
    unknown_count = (df[column] == 'unknown').sum()

    print(f"{column}: {unknown_count}")


### 4E. Numerical Value Range Audit

#### 4E.1 Generate Numerical Summary

In [ ]:
df[numerical_columns].describe().T

#### 4E.2 Check Maximum Value Explicitly

In [ ]:
df[numerical_columns].max()

#### 4E.3 Check Minimum Values

In [ ]:
df[numerical_columns].min()

## STEP 5 Data Cleaning Strategy & Business Handling Logic

### 5A. Create a working copy of Dataset

In [ ]:
df_clean = df.copy()

### 5B. Standardize Column Names

In [ ]:
df_clean.columns = (
    df_clean.columns
    .str.lower()
    .str.replace('.', '_')
    .str.replace(' ', '_')
)

### 5C. columns name checking

In [ ]:
df_clean.columns

### 5D. Strategic Handling of "Unknown" Values

To preserve analytical integrity, `"unknown"` values will be treated as a separate business category rather than being removed or imputed during cleaning.

This is a strategic analytical decision because unknown values may represent:

* incomplete customer profiling
* operational gaps in data collection
* customer unwillingness to disclose information

These characteristics themselves may influence customer conversion behavior.

---

#### 📌 Retention Strategy

Fields containing `"unknown"` values — such as `education`, `housing_loan_status`, and `personal_loan_status` — will be retained during the initial analysis phase.

This approach helps:

* preserve operational realism
* avoid introducing bias through aggressive imputation
* investigate whether incomplete customer profiles behave differently from fully profiled customers

---

#### 🔍 Additional Validation Rules

* True null values (if detected later) will be investigated individually
* Extreme numerical values will be audited before any filtering decisions are made
* Outliers will be evaluated from a business perspective before removal

---

#### 📊 Analyst Note

During exploratory analysis, we will compare conversion behavior between **“Known”** and **“Unknown”** customer profiles to determine whether data completeness itself influences subscription outcomes.


### 5E. Recheck Dataset Integrity

In [ ]:
df_clean.shape

### 5F. Validate Data Types Again

In [ ]:
df_clean.info()

### 5E. Semantic Business Renaming

In [ ]:
# Renaming columns for business readability

df_clean = df_clean.rename(columns={
    'emp_var_rate' : 'employment_variation_rate',
    'cons_price_idx' : 'consumer_price_index',
    'cons_conf_idx': 'consumer_confidence_index',
    'euribor3m' : 'market_interest_rate',
    'nr_employed' : 'employment_level',
    'y' :  'subscribed',
    'default' : 'credit_default_status',
    'housing' : 'housing_loan_status',
    'loan' : 'personal_loan_status',
    'contact' : 'contact_method',
    'duration' : 'call_duration_seconds',
    'campaign' : 'current_campaign_contacts',
    'pdays' : 'days_since_previous_contact',
    'previous' : 'previous_campaign_contacts',
    'poutcome' : 'previous_campaign_outcome',
    'month' : 'campaign_month',
    'day_of_week' : 'campaign_day_of_week'
    
})


### 5F Validating Column Names Again

In [ ]:
df_clean.info()

## Step 6 Feature Understanding & Analytical Preparation

### 6A. Identify target variable Distribution

In [ ]:
df_clean['subscribed'].value_counts()

### 6B. Check target variable percentage

In [ ]:
df_clean['subscribed'].value_counts(normalize=True) * 100

### 6C. Identify Core Analytical Feature Groups

In [ ]:
# --- Definitions Cell ---
customer_demographics = ['age', 'job', 'marital', 'education']
financial_features = ['credit_default_status', 'housing_loan_status', 'personal_loan_status']
campaign_features = ['contact_method', 'campaign_month', 'campaign_day_of_week', 'call_duration_seconds', 'current_campaign_contacts']
historical_features = ['days_since_previous_contact', 'previous_campaign_contacts', 'previous_campaign_outcome']
economic_features = ['employment_variation_rate', 'consumer_price_index', 'consumer_confidence_index', 'market_interest_rate', 'employment_level']


### 6D. Identifying High-Priority Variables for Investigation

Based on the project objective, I’ve identified a subset of variables likely to have the strongest influence on customer conversion behavior. These variables will become the primary focus of the initial EDA phase.

#### 🔍 Key Variables & Analytical Rationale

* **`call_duration_seconds`**
  Likely the strongest proxy for customer engagement and interest. We need to verify whether longer conversations genuinely increase conversion probability or simply reflect extended but unsuccessful interactions.

* **`current_campaign_contacts`**
  Essential for identifying the point of diminishing returns. This analysis will help determine when campaign persistence begins creating customer fatigue.

* **`previous_campaign_outcome`**
  Historical campaign success is expected to strongly influence future conversion behavior. We will investigate how previous positive outcomes impact current subscription likelihood.

* **`market_interest_rate`**
  Represents the broader interest-rate environment during the campaign period. This will help evaluate whether economic conditions influenced customer willingness to subscribe to term deposits.

* **`age` and `job`**
  These variables will drive demographic segmentation analysis. Certain age groups and occupational categories may demonstrate stronger affinity toward banking investment products.

---

> 📌 Analyst Note: These are not validated drivers yet. They represent initial business hypotheses that will be tested through exploratory analysis and statistical investigation.


## 6E. Business Hypothesis Planning

Before moving into the visualization phase, I am establishing the following hypotheses based on the business objective and the high-priority variables identified in the previous step.

#### 🔍 Core Hypotheses to Test

* **H1: Engagement Influence**

  * **Hypothesis:** Longer `call_duration_seconds` are expected to be associated with higher subscription rates.
  * **Logic:** Longer conversations may indicate that the agent successfully moved beyond the initial pitch and generated stronger customer engagement.

* **H2: The Fatigue Threshold**

  * **Hypothesis:** Conversion rates may decline beyond a certain contact threshold within `current_campaign_contacts`.
  * **Logic:** Excessive contact attempts could lead to customer fatigue, reduced responsiveness, and diminishing campaign effectiveness.

* **H3: Economic Sensitivity**

  * **Hypothesis:** Higher `market_interest_rate` environments may influence customer willingness to subscribe to term deposits.
  * **Logic:** Broader interest-rate conditions can impact customer investment preferences and financial decision-making behavior.

* **H4: Previous Success Bias**

  * **Hypothesis:** Customers with a previous `"success"` outcome in `previous_campaign_outcome` are expected to show significantly higher conversion rates than new prospects.
  * **Logic:** Prior positive experiences and existing customer trust often influence future banking behavior.

---

### 🎯 Target Variable (Primary KPI)

* **Variable:** `subscribed`
* **Definition:** A binary indicator representing whether the customer accepted the term deposit offer. This serves as the primary outcome variable for evaluating customer conversion behavior and the final metric against which all hypotheses above will be tested.


## STEP 7 — Univariate Analysis

### Objective
Analyze individual variables independently to understand customer composition, campaign structure, engagement behavior, and overall business distributions before relationship analysis begins.

### 7A. Subscription Outcome Distribution

This analysis establishes the baseline conversion performance of the campaign by evaluating the distribution of customer subscription outcomes.

### 7A. Subscription Outcome Distribution

This analysis establishes the baseline conversion performance of the campaign by evaluating the distribution of customer subscription outcomes.

In [ ]:
# Display subscription outcome counts

df_clean['subscribed'].value_counts()

#### 📌 Initial Observation

The dataset shows a strong imbalance between subscribed and non-subscribed customers, indicating that successful conversions represent a relatively small portion of the campaign population. This behavior is realistic for direct banking campaigns.

In [ ]:
# Display subscription percentage distribution

(
    df_clean['subscribed']
    .value_counts(normalize=True) * 100
)

#### 📌 Business Interpretation

The subscription rate established here becomes the baseline KPI for evaluating all future segmentation, behavioral, and campaign effectiveness analysis.


### 7B. Customer Demographic Analysis

The following analysis explores the demographic structure of the customer base to identify dominant customer groups and future segmentation opportunities.


In [ ]:
# Display customer distribution by occupation

df_clean['job'].value_counts()

#### 📌 Initial Observation

The customer base appears concentrated within a few occupational categories, suggesting potential demographic clustering within the campaign population.


In [ ]:
# Display customer distribution by education level

df_clean['education'].value_counts()

#### 📌 Initial Observation

Education distribution provides insight into the socioeconomic structure of the campaign audience. The presence of `"unknown"` values may also indicate incomplete customer profiling.


In [ ]:
# Display customer distribution by marital status

df_clean['marital'].value_counts()

#### 📌 Initial Observation

Marital status distribution helps understand household composition and customer lifecycle structure within the banking customer base.


### 7C. Age Distribution Analysis 
Age is one of the most important demographic variables for customer segmentation and behavioural analysis

In [ ]:
# Generate discriptive statistic for age 
df_clean['age'].describe()

In [ ]:
# Plot Customer age distribution 
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")
sns.histplot(data = df, x ='age', bins = 30, kde = True, color = 'skyblue')
plt.title('Customer Age Distribution', fontsize = 14)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Number of Customer', fontsize = 12)
plt.tight_layout()
plt.show()

#### 📌 Initial Observation

The age distribution will help identify dominant customer age brackets, potential outlier groups, and lifecycle segmentation opportunities.


### 7D. Campaign Contact Analysis
This section evaluates campaign contact intensity and customer engagement behaviour during the marketing campaign

In [ ]:
# Generate descriptive statistics for campaign contacts
df_clean['current_campaign_contacts'].describe()

#### 📌 Initial Observation

Large variation in campaign contact frequency may indicate inconsistent outreach strategies and potential customer fatigue in heavily contacted segments.


In [ ]:
# Generate descriptive statistics for call duration

df_clean['call_duration_seconds'].describe()

#### 📌 Initial Observation

Call duration variability may reflect differences in customer engagement levels, interest intensity, and conversation quality during campaign interactions.


### 7E. Historical Engagement Analysis
Historical engagement variables provide insights into the continuity of customer relationships and previous campaign interaction

In [ ]:
# Display previous campaign outcome distribution 
df_clean['previous_campaign_outcome'].value_counts()

#### 📌 Initial Observation

A high proportion of `"nonexistent"` outcomes would indicate that many customers were being contacted for the first time, while successful previous outcomes may signal stronger relationship continuity.


## 📊 Summary of Univariate Analysis

The univariate analysis phase establishes a foundational understanding of:

* customer demographics
* campaign structure
* engagement behavior
* historical relationship patterns
* baseline subscription performance

These findings will guide the next phase of analysis focused on identifying variables associated with customer conversion behavior.


# STEP 8 — Bivariate & Conversion Analysis

## Objective

Analyze the relationship between customer features and the subscription outcome in order to identify patterns associated with conversion behavior.

Unlike univariate analysis, this phase focuses on understanding:

* which customer groups convert more frequently
* which campaign conditions influence subscription behavior
* which variables show strong behavioral separation

---

### 8A. Conversion Rate by Job Category

This analysis evaluates whether customer occupation influences subscription behavior and campaign responsiveness.


In [ ]:
# calculate conversion rate by occupation

(
    df_clean
    .groupby('job')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100

).round(2)

#### 📌 Initial Observation

Occupational categories may exhibit significantly different conversion behaviors, helping identify customer segments with stronger affinity toward banking products.


### 8B. Conversion Rate by Education Level

This analysis investigates whether educational background influences customer engagement and subscription outcomes.


In [ ]:
# calculate conversion rate by education level

(
    df_clean
    .groupby('education')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)
    

#### 📌 Initial Observation

Education levels may reveal differences in financial awareness, investment behavior, and responsiveness to banking campaigns.


### 8C. Conversion Rate by Marital Status

This analysis evaluates whether household structure influences subscription behavior.


In [ ]:
# Calculate conversion rate by marital status
(
    df_clean
    .groupby('marital')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Different marital groups may demonstrate varying financial priorities and banking engagement patterns.


### 8D. Conversion Rate by Contact Method

This analysis evaluates the effectiveness of different communication channels used during the campaign.


In [ ]:
# Calculate conversion rate by contact method
(
    df_clean
    .groupby('contact_method')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

### 8E. Conversion Rate by Previous Campaign Outcome

This analysis investigates whether historical campaign success influences future subscription probability.


In [ ]:
# Calculate conversion rate by previous campaign outcome

(
    df_clean
    .groupby('previous_campaign_outcome')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Customers with previously successful interactions may demonstrate significantly stronger conversion behavior due to existing trust and relationship continuity.


### 8F. Age vs Subscription Analysis

This analysis evaluates whether customer age influences subscription behavior.


In [ ]:
# Compare age distribution by subscription outcome
df_clean.groupby('subscribed')['age'].describe()

#### 📌 Initial Observation

Differences in age distribution between subscribed and non-subscribed customers may reveal lifecycle-based behavioral patterns.


### 8G. Campaign Contact Frequency vs Subscription

This analysis investigates whether repeated campaign contact attempts influence conversion probability.


In [ ]:
# Compare campaign contact behaviour by subscription outcome
df_clean.groupby('subscribed')['current_campaign_contacts'].describe()

### 8H. Call Duration vs Subscription

This analysis evaluates the relationship between customer interaction duration and subscription outcomes.


In [ ]:
# Compare call duration behaviour by subscription outcome
df_clean.groupby('subscribed')['call_duration_seconds'].describe()

#### 📌 Initial Observation

Longer conversations may indicate stronger customer engagement and increased interest in banking products.


## 📊 Summary of Bivariate Analysis

The bivariate analysis phase begins identifying variables associated with customer conversion behavior by comparing customer segments, campaign conditions, and engagement patterns against the subscription outcome.

Key investigation areas include:

* demographic conversion behavior
* communication effectiveness
* historical relationship influence
* campaign fatigue patterns
* engagement intensity differences

These findings will guide the next stage focused on deeper behavioral and segmentation analysis.


# STEP 9 — Segmentation & Behavioral Investigation

## Objective

Identify high-performing and underperforming customer segments by combining demographic, financial, campaign, and historical engagement variables.

This phase moves beyond generic variable comparison and focuses on:

* identifying strategically valuable customer groups
* detecting campaign inefficiencies
* understanding behavioral engagement patterns
* uncovering business targeting opportunities



### 9A. Age Group Segmentation

Customer age is segmented into meaningful lifecycle categories to investigate whether specific age groups demonstrate stronger subscription behavior.

This segmentation helps transform raw demographic data into interpretable business cohorts.


In [ ]:
# create age group categories

df_clean['age_group'] = pd.cut(

    df_clean['age'],

    bins = [17, 25, 35, 45, 55, 65, 100],

    labels=[
        '18-25',
        '26-35',
        '36-45',
        '46-55',
        '56-65',
        '65+'
    ]
)
# Display age group distribution
df_clean['age_group'].value_counts()


#### 📌 Initial Observation

Age segmentation allows the analysis to identify which lifecycle stages demonstrate the strongest banking engagement and subscription behavior.


### 9B. Conversion Rate by Age Group

This analysis evaluates whether customer conversion behavior differs across lifecycle segments.



In [ ]:
# Calculate conversion rate by age group

(
    df_clean
    .groupby('age_group')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Certain age groups may demonstrate stronger affinity toward term deposit products due to financial stability, investment preferences, or long-term savings behavior.



### 9C. Campaign Fatigue Investigation

This analysis investigates whether excessive contact attempts reduce campaign effectiveness and customer responsiveness.



In [ ]:
# Analyse average campaign contacts by subscription outcome

df_clean.groupby('subscribed')['current_campaign_contacts'].mean().round()

#### 📌 Initial Observation

Higher contact frequency among non-subscribed customers may indicate diminishing campaign returns and the presence of customer fatigue.



### 9D. High-Value Customer Segment Investigation

This analysis focuses on strategically important occupational segments that are more likely to represent financially stable or economically active customers.

Rather than evaluating the entire customer base uniformly, this step investigates whether high-value professional categories demonstrate stronger subscription behavior and campaign responsiveness.

The selected customer segments include:

* management
* technician
* admin.
* blue-collar

These groups represent a significant portion of the campaign population and may exhibit different financial engagement patterns based on income stability, employment structure, and banking needs.



In [ ]:
# Define strategically important occupational segments

high_value_jobs = [

    'management',
    'technician',
    'blue-collar',
    'admin.'
]

# Analyze subscription behavior within high-value customer segments

(
    df_clean[
        df_clean['job'].isin(high_value_jobs)
    ]

    .groupby('job')['subscribed']

    .value_counts(normalize=True)

    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Differences in conversion behavior across these occupational segments may help identify customer groups with stronger affinity toward banking investment products.

This analysis also supports future strategic targeting decisions by highlighting which customer profiles respond more positively to campaign outreach.


### 9E. New vs Existing Customer Investigation

This analysis compares first-time prospects against previously contacted customers to evaluate whether relationship continuity influences subscription behavior.


In [ ]:
# create prospect calssification

df_clean['customer_relationship_type'] = np.where(
    df_clean['previous_campaign_contacts'] == 0,

    'New Prospects',
    'Previously Contacted'
)
# Analyse conversion behaviour by relationship type
(
    df_clean
    .groupby('customer_relationship_type')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Previously contacted customers may demonstrate stronger conversion behavior due to existing familiarity, trust, or prior campaign engagement.


### 9F. Communication Channel Effectiveness

This analysis evaluates whether campaign communication channels influence customer conversion behavior.


In [ ]:
# Analyze subscription behaviour by communication channel
(
    df_clean
    .groupby('contact_method')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Significant performance differences between communication channels may reveal operational inefficiencies and opportunities for campaign optimization.


### 9G. Historical Campaign Success Investigation

This analysis evaluates whether customers with successful historical campaign outcomes demonstrate stronger future conversion behavior.


In [ ]:
# Analyse historical campaign outcomes against subscription behaviour
(
    df_clean
    .groupby('previous_campaign_outcome')['subscribed']
    .value_counts(normalize=True)
    .unstack() * 100
).round(2)

#### 📌 Initial Observation

Customers with previous successful interactions may exhibit stronger trust, engagement continuity, and responsiveness toward future banking campaigns.


### 9H. Economic Environment Sensitivity Analysis

This analysis investigates whether external economic conditions influence customer subscription behavior.


In [ ]:
# Compare economic indicators by subscription outcome
(
    df_clean
    .groupby('subscribed')[[
        'market_interest_rate',
        'consumer_confidence_index',
        'employment_variation_rate'
    ]]
    .mean()
).round(2)

#### 📌 Initial Observation

Differences in economic conditions between subscribed and non-subscribed customers may indicate that broader financial environments influence investment decision-making behavior.


## 📊 Summary of Segmentation & Behavioral Investigation

This phase transitions the project from generic exploratory analysis toward strategic business intelligence.

The investigation focused on:

* lifecycle-based customer behavior
* campaign fatigue signals
* high-value customer segments
* relationship continuity effects
* communication effectiveness
* economic sensitivity patterns

These findings will later support:

* KPI engineering
* SQL business analysis
* dashboard storytelling
* strategic customer targeting recommendations
* executive decision-support analysis


# STEP 10 — Derived Analytical Feature Engineering

## Objective

Create business-driven analytical features that enhance segmentation capability, behavioral interpretation, and dashboard intelligence.

Unlike raw dataset variables, derived features are engineered using patterns observed during exploratory analysis and strategic investigation.

This phase focuses on transforming:

* raw campaign behavior
* customer interaction patterns
* relationship continuity
* demographic structure

into higher-value analytical dimensions.


### 10A. Contact Intensity Segmentation

This feature categorizes customers based on the number of campaign contact attempts received.


In [ ]:
# Create campaign contact intensity segments
df_clean['contact_intensity_segment'] = pd.cut(
    
    df_clean['current_campaign_contacts'],

    bins = [0, 2, 5, 10, 100],

    labels=[
        'Low Contact',
        'Moderate Contact',
        'High Contact',
        'Extreme Contact'
    ]
)
# Display segment distribution

df_clean['contact_intensity_segment'].value_counts()

#### 📌 Business Value

This feature supports campaign fatigue analysis and helps identify the point where repeated outreach begins reducing effectiveness.


### 10C. Age Lifecycle Segmentation

This feature converts raw customer age into business-readable lifecycle categories for strategic demographic analysis.


In [ ]:
# Create customer lifecycle age groups

df_clean['customer_lifecycle_segment'] = pd.cut(

    df_clean['age'],

    bins=[17, 25, 35, 45, 55, 65, 100],

    labels =[
        'Young Adult',
        'Early Career',
        'Mid Career',
        'Experienced Professional',
        'Pre-Retirement',
        'Retirement Age'

    ]
)
# Display lifecycle distribution

df_clean['customer_lifecycle_segment'].value_counts()

### 10D. Customer Profile Completeness Indicator

This feature identifies customers with incomplete profile information based on the presence of `"unknown"` values.


In [ ]:
# Create profile completeness indicator

df_clean['profile_completeness'] = np.where(

    (
        (df_clean['education'] == 'unknown') |
        
        (df_clean['housing_loan_status'] == 'unknown') |
        
        (df_clean['personal_loan_status'] == 'unknown')

    ),
    'Incomplete Profile',

    'Complete Profile'

)
# Display Profile completeness distribution

df_clean['profile_completeness'].value_counts()

#### 📌 Business Value

This feature allows investigation into whether incomplete customer profiling influences conversion behavior and campaign responsiveness.


### 10E. Engagement Duration Category

This feature segments customer interactions based on call duration intensity.


In [ ]:
# Create engagement duration segments

df_clean['engagement_duration_segment'] = pd.cut(
    
    df_clean['call_duration_seconds'],

    bins = [0, 120, 300, 600, 5000],

    labels = [
        'Short Interaction',
        'Moderate Interaction',
        'Long Interaction',
        'Highly Engaged'
    ]
)

# Display engagement segment distribution

df_clean['engagement_duration_segment'].value_counts()

#### 📌 Business Value

This feature helps identify customer engagement intensity and supports future interaction quality analysis.


### 10F. Previous Engagement Status

This feature classifies customers based on historical campaign interaction outcomes.


In [ ]:
# Create previous engagement status

df_clean['historical_engagement_status'] = np.where(

    df_clean['previous_campaign_outcome'] == 'success',

    'Previously Sucessfull',
    
    'No Previous Success'
)

# Display historical engagement distribution

df_clean['historical_engagement_status'].value_counts()

## 📊 Summary of Derived Feature Engineering

This phase transformed raw operational variables into business-readable analytical dimensions designed to improve:

* segmentation quality
* dashboard storytelling
* behavioral investigation
* KPI development
* executive interpretation

The engineered features created during this phase will become foundational components for:

* SQL business analysis
* Power BI modeling
* DAX measures
* strategic customer segmentation
* stakeholder-facing dashboard intelligence


# STEP 11 — Correlation & Statistical Investigation

## Objective

Validate whether the patterns observed during exploratory analysis are supported by measurable statistical relationships.

This phase moves beyond visual and descriptive analysis by investigating:

* variable relationships
* behavioral strength
* numerical associations
* potential analytical drivers of conversion behavior

The goal is not to build predictive models, but to strengthen analytical confidence in the business patterns identified earlier.


### 11A. Numerical Correlation Analysis

This analysis evaluates relationships between numerical variables to identify potential behavioral associations and operational dependencies.


In [ ]:
# Select numerical columns 

numerical_columns = [
    'age',
    'call_duration_seconds',
    'current_campaign_contacts',
    'days_since_previous_contact',
    'previous_campaign_contacts',
    'employment_variation_rate',
    'consumer_price_index',
    'consumer_confidence_index',
    'market_interest_rate',
    'employment_level'
]

# Generate correlation matrix

df_clean[numerical_columns].corr().round(2)
    

#### 📌 Initial Observation

Strong positive or negative correlations may indicate operational relationships, behavioral dependencies, or macroeconomic alignment between variables.


### 11B. Correlation Heatmap Visualization

A heatmap improves interpretability by visually highlighting the strength and direction of numerical relationships.


In [ ]:
# Plot correlation heatmap

import seaborn as sns
import pandas as pd

plt.figure(figsize=(12, 8))

sns.heatmap(

    df_clean[numerical_columns].corr(),

    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')

plt.show()

#### 📌 Initial Observation

Variables showing unusually strong correlation may require deeper business investigation to understand whether the relationship is operational, economic, or behavioral in nature.


### 11C. Subscription Encoding for Statistical Analysis

To evaluate statistical relationships with the subscription outcome, the target variable must first be converted into numerical format.


In [ ]:
# Encode subscription outcome

df_clean['subscription_encoded'] = np.where(
    
    df_clean['subscribed'] == 'yes',

    1,

    0

)
# Validate encoded distribution

df_clean['subscription_encoded'].value_counts()
    

#### 📌 Initial Observation

Encoding the target variable enables correlation analysis and future statistical comparisons involving customer conversion behavior.


### 11D. Numerical Variables vs Subscription Correlation

This analysis evaluates which numerical variables demonstrate the strongest measurable relationship with customer subscription behavior.


In [ ]:
# Analyse correlation with subscription outcome

df_clean[
    numerical_columns + ['subscription_encoded']

].corr()['subscription_encoded'].sort_values(ascending=False)

#### 📌 Initial Observation

Variables showing stronger correlation with subscription behavior may represent important operational or behavioral signals influencing campaign outcomes.


### 11E. Contact Frequency Statistical Investigation

This analysis investigates whether campaign contact intensity differs meaningfully between subscribed and non-subscribed customers.


In [ ]:
# Compare average campaign contacts

df_clean.groupby('subscribed')[
    'current_campaign_contacts'
].mean().round(2)

#### 📌 Initial Observation

Large differences in average contact frequency may strengthen evidence for campaign fatigue effects identified during earlier analysis phases.


### 11F. Call Duration Statistical Investigation

This analysis evaluates whether customer engagement duration differs significantly between subscription outcomes.


In [ ]:
# Compare average call duration

df_clean.groupby('subscribed')[
    'call_duration_seconds'

].mean().round(2)

#### 📌 Initial Observation

Significant differences in call duration may reinforce the hypothesis that stronger engagement intensity is associated with higher conversion behavior.


## 📊 Summary of Correlation & Statistical Investigation

This phase strengthened analytical confidence by validating whether observed behavioral patterns also demonstrate measurable statistical relationships.

The investigation focused on:

* numerical variable relationships
* conversion-related correlations
* engagement intensity signals
* campaign fatigue indicators
* macroeconomic influence patterns

These findings will help guide:

* SQL analytical modeling
* dashboard KPI selection
* executive storytelling
* strategic business interpretation


# STEP 12 — Business Insight Extraction & KPI Framework

## Objective

Transform analytical findings into structured business intelligence that can later drive:

* SQL analysis
* dashboard storytelling
* executive reporting
* KPI monitoring
* strategic decision-making

This phase focuses on converting raw analytical observations into measurable business insights and operational performance indicators.


### 12A. Core Conversion KPI Definition

The subscription outcome will serve as the primary business KPI throughout the project.


In [ ]:
# Calculate overall subscription rate 

overall_conversion_rate = (

    df_clean['subscribed']
    .value_counts(normalize=True)['yes'] * 100
)
print(
    f"Overall Subscription Rate : {overall_conversion_rate:.2f}%"
)

#### 📌 Business Insight

This KPI establishes the baseline campaign conversion performance against which all future segmentation and operational effectiveness analysis will be evaluated.


### 12B. High-Converting Customer Segments

This analysis identifies customer groups demonstrating stronger subscription behavior.


In [ ]:
# Identify highest converting job segments
(
    df_clean

    .groupby('job')['subscription_encoded']

    .mean()

    .sort_values(ascending=False) * 100

).round(2).head(10)

### 12C. Campaign Fatigue KPI

This analysis evaluates whether excessive contact attempts negatively influence conversion performance.


In [ ]:
# Analyse contact frequency by subscription outcome

df_clean.groupby('subscribed')[

    'current_campaign_contacts'

].mean().round(2)

#### 📌 Business Insight

If non-subscribed customers receive significantly more contact attempts, this may indicate inefficient outreach strategies and campaign fatigue effects.


### 12D. Engagement Quality KPI

This analysis investigates whether longer customer interactions are associated with stronger conversion performance.


In [ ]:
# Analyse average call duration by subscription outcome

df_clean.groupby('subscribed')[
    'call_duration_seconds'
].mean().round(2)

#### 📌 Business Insight

Longer interaction duration may indicate stronger customer engagement, better conversation quality, and increased interest in banking products.


### 12E. Customer Relationship Continuity KPI

This analysis evaluates whether previously engaged customers convert more effectively than first-time prospects.


In [ ]:
# Analyse relationship continuity impact

(
    df_clean

    .groupby('customer_relationship_type')['subscription_encoded']

    .mean() * 100

).round(2)

#### 📌 Business Insight

Existing customer familiarity and prior engagement history may significantly improve campaign responsiveness and conversion probability.


### 12F. Profile Completeness KPI

This analysis investigates whether incomplete customer profiles influence conversion behavior.


In [ ]:
# Analyse profile completeness impact

(
    df_clean

    .groupby('profile_completeness')['subscription_encoded']

    .mean() * 100

).round(2)

### 12G. Strategic KPI Identification

Based on the exploratory and statistical analysis phases, the following KPIs have been identified as strategically important for dashboard development:

#### 📊 Primary KPIs

* Overall Subscription Rate
* Campaign Conversion Rate
* Customer Engagement Rate
* Average Call Duration
* Contact Frequency

#### 👥 Customer Intelligence KPIs

* Conversion by Age Group
* Conversion by Occupation
* Conversion by Relationship Type
* Conversion by Profile Completeness

#### 📞 Campaign Performance KPIs

* Contact Fatigue Indicators
* Communication Channel Effectiveness
* Historical Campaign Success Rate
* Engagement Duration Segmentation

#### 🌍 Economic Context KPIs

* Conversion vs Interest Rate Environment
* Consumer Confidence Impact
* Employment Trend Influence


## 📊 Summary of Business Insight Extraction

This phase transformed analytical observations into structured business intelligence and measurable KPI frameworks.

The investigation identified:

* high-performing customer segments
* operational inefficiencies
* customer engagement patterns
* relationship continuity effects
* campaign fatigue signals
* strategic dashboard KPIs

These outputs will directly support the next project stages:

* SQL business analysis
* Power BI data modeling
* DAX calculations
* executive dashboard storytelling
* stakeholder-facing business intelligence development


# STEP 13 — Statistical Hypothesis Testing

## Objective

Validate whether the key behavioral patterns identified during exploratory analysis are statistically significant rather than random variation.

This phase focuses on testing the project’s most important business hypotheses related to:

* campaign fatigue
* customer engagement
* relationship continuity

The goal is to strengthen analytical confidence and support business conclusions with statistical evidence.


### 13A. Contact Fatigue Hypothesis Test

#### Mann–Whitney U Test

This test evaluates whether the distribution of campaign contact frequency differs significantly between subscribed and non-subscribed customers.

#### 📌 Business Hypothesis

Customers receiving excessive contact attempts may demonstrate lower conversion behavior due to campaign fatigue.

#### 📌 Why Mann–Whitney?

The variable `current_campaign_contacts` is highly skewed and contains outliers, making a non-parametric test more appropriate than a standard t-test.


In [ ]:
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare customer groups

group_success = df_clean[df_clean['subscribed'] == 'yes']['current_campaign_contacts']

group_failure = df_clean[df_clean['subscribed'] == 'no']['current_campaign_contacts']

# Perform Mann_Whitney U Test
stat, p_value = mannwhitneyu( group_success, group_failure,  alternative='two-sided' )

# Display statistical results
print(f"Mann-Whitney U Statistic: {stat}")

print(f"P-value: {p_value}")

# Statistical interpretation

alpha = 0.05

if p_value < alpha:

    print("\nResult: Statistically Significant")

    print("Conclusion: Contact frequency differs significantly between conversion groups")

else: 
    print("\nResult: Nor Statistically Significant")

    print("Conclusion: Contact frequency differences may be due to random variation.")

# Visualization

plt.figure(figsize=(10,6))

sns.boxplot( x ='subscribed', y='current_campaign_contacts', data = df_clean,  showfliers=False)

plt.title('Campaign Contact Frequency by Subscription Outcome')

plt.xlabel('Subscription Outcome')

plt.ylabel('Number of Campaign Contacts')

plt.show()

#### 📌 Statistical Finding

The Mann–Whitney U Test produced a highly significant p-value (`p < 0.05`), indicating that campaign contact frequency differs meaningfully between subscribed and non-subscribed customers.

This result strengthens the campaign fatigue hypothesis and suggests that contact intensity influences customer conversion behavior.

The analysis indicates that repeated outreach attempts may reduce campaign effectiveness beyond a certain threshold, potentially leading to lower responsiveness among heavily contacted customers.

From a business perspective, this finding highlights the importance of optimizing contact strategy rather than relying on excessive outreach frequency.


### 13A.1 Contact Fatigue Threshold Analysis

This analysis investigates how customer conversion rates change as campaign contact frequency increases.

The objective is to identify the operational point at which additional outreach attempts begin reducing campaign effectiveness.

In [ ]:
# Calculate conversion rate by contact frequency

contact_threshold_analysis = (

    df_clean

    .groupby('current_campaign_contacts')['subscription_encoded']

    .mean()

    .reset_index()
)

# Convert to percentage

contact_threshold_analysis['subscription_encoded'] = (

    contact_threshold_analysis['subscription_encoded'] * 100

)

contact_threshold_analysis = contact_threshold_analysis[
                    contact_threshold_analysis['current_campaign_contacts'] <= 15
]

# Display results

contact_threshold_analysis.head(15)

In [ ]:
# Plot conversion trend by contact frequency

plt.figure(figsize=(12, 6))

sns.lineplot(

    data=contact_threshold_analysis,

    x = 'current_campaign_contacts',

    y = 'subscription_encoded',

    marker='o'
)

plt.title('Conversion Rate by Campaign Contact Frequency')

plt.xlabel('Number of Campaign Contacts')

plt.ylabel('Subscription Rate (%)')

plt.xticks(range(1, 16))

plt.grid(True)

plt.show()

The analysis suggests that conversion effectiveness begins declining substantially after approximately 5–8 contact attempts.

Beyond this range, subscription performance deteriorates sharply and becomes increasingly unstable, indicating the presence of campaign fatigue and diminishing outreach returns.

### 13B. Engagement Duration Hypothesis Test

#### Independent T-Test

This test evaluates whether subscribed customers have significantly different call durations compared to non-subscribed customers.

#### 📌 Business Hypothesis

Longer customer interactions are associated with stronger engagement and higher subscription probability.

#### 📌 Why T-Test?

The Independent T-Test compares the average values (means) between two independent customer groups:

* subscribed customers
* non-subscribed customers

This helps determine whether the observed engagement difference is statistically meaningful rather than random variation.


In [ ]:
# Import required libraries

from scipy.stats import ttest_ind

import matplotlib.pyplot as plt

import seaborn as sns

# Prepare customer groups

group_success = df_clean[df_clean['subscribed'] == 'yes']['call_duration_seconds']

group_failure = df_clean[df_clean['subscribed'] == 'no']['call_duration_seconds']

# Perform Independent T-Test

t_statistic, p_value = ttest_ind(

    group_success,

    group_failure,

    equal_var=False
)

# Display statistical result

print(f"T-statistic: {t_statistic}")

print(f"P-value: {p_value}")

#statistical interpretation

alpha = 0.05

if p_value < alpha:
    print("\nResult: statistically Significant")

    print("Conclusion: Call duration differs significantly between subscribed and non-subscribed customers")

else:

    print("\n: Result: Not Statistically Significant")


    print("Conclusion: Observed call differences may be due to random variation")

In [ ]:
# Visualization

plt.figure(figsize=(10, 6))

sns.boxplot( x='subscribed', y='call_duration_seconds', data=df_clean, showfliers=False)

plt.title('Call duration by subscription outcome')

plt.xlabel('Subscription Outcome')

plt.ylabel('Call Duration (Seconds)')

plt.show()


#### 📌 Statistical Finding

The Welch’s Independent T-Test produced a highly significant result (`p < 0.05`), indicating that call duration differs meaningfully between subscribed and non-subscribed customers.

The boxplot visualization further reinforces this finding by showing substantially higher median engagement duration among subscribed customers compared to non-subscribed customers.

Customers who subscribed generally participated in longer and more variable conversations, suggesting deeper engagement, stronger product interest, and more meaningful interaction quality during campaign calls.

From a business perspective, this analysis indicates that sustained customer engagement is strongly associated with successful conversion behavior, making call duration one of the most influential behavioral indicators observed in the project.


### 13B.1 Engagement Threshold Analysis

This analysis investigates how customer conversion behavior changes across different call duration ranges.

The objective is to identify the approximate engagement duration threshold at which customers begin demonstrating stronger subscription probability.

This helps determine whether longer interactions represent meaningful customer interest rather than simple conversation length.


In [ ]:
# Create call duration segment

df_clean['call_duration_group'] = pd.cut(

    df_clean['call_duration_seconds'],

    bins = [0, 60, 120, 300, 600, 5000],

    labels=[
        '0-1 min',
        '1-2 min',
        '2-5 min',
        '5-10 min',
        '10+ min'
    ]
)
# calculate conversion rate by duration group

duration_threshold_analysis = (

    df_clean

    .groupby('call_duration_group')['subscription_encoded']

    .mean()

    .reset_index()

)

# Convert to percentage

duration_threshold_analysis['subscription_encoded'] = (

    duration_threshold_analysis['subscription_encoded'] * 100

)

# Display results
print(duration_threshold_analysis)

In [ ]:
# Plot conversion rate by engagement duration

plt.figure(figsize=(10, 6))

sns.barplot(

    data = duration_threshold_analysis,

    x = 'call_duration_group',

    y = 'subscription_encoded'

)

# Chart formatting

plt.title('Conversion Rate by Call Duration')

plt.xlabel('Call Duration Group')

plt.ylabel('subscription Rate(%)')

plt.show()



#### 📌 Business Finding

The analysis indicates that customer conversion probability begins increasing meaningfully after approximately 2–5 minutes of engagement.

Calls lasting less than 2 minutes demonstrate extremely low subscription rates, suggesting limited customer interest or early rejection behavior.

A substantial increase in conversion performance is observed beyond the 5-minute mark, while interactions exceeding 10 minutes demonstrate the highest conversion likelihood.

This pattern suggests that sustained customer engagement is a strong behavioral indicator of successful conversion outcomes.


### 13C. Previous Success vs Subscription Test

#### Chi-Square Test of Independence

This test evaluates whether previous campaign outcomes are associated with future subscription behavior.

#### 📌 Business Hypothesis

Customers who previously responded successfully to banking campaigns are more likely to subscribe again in future campaigns due to existing trust, familiarity, and positive engagement history.

#### 📌 Why Chi-Square?

Both variables involved in this analysis are categorical:

* `previous_campaign_outcome`
* `subscribed`

The Chi-Square Test helps determine whether these variables are statistically associated or behave independently.


In [ ]:
# Import required libraries

from scipy.stats import chi2_contingency

import matplotlib.pyplot as plt

import seaborn as sns

# Create contingency table

contingency_table = pd.crosstab(
    
    df_clean['previous_campaign_outcome'],

    df_clean['subscribed']
)

# Perform Chi-Square Test

chi2_statistic, p_value, dof, expected = chi2_contingency(
    
    contingency_table
)

# Display statistical results

print(f"Chi-Square Statistc: {chi2_statistic}")

print(f"P-value: {p_value}")

# Statistical interpretation

alpha = 0.05

if p_value < alpha:

    print("\nResult: Statistically Significant")

    print("Conclusion: Previous campaign outcome and subscription behaviour are statistically associated.")

else:
    print("\nResult: Not Statistically Significant")

    print("Conclusion: No significant association between previous campaign outcomes and subscription behaviour.")    

In [ ]:
# Visualization

subscription_rate = (
    df_clean

    .groupby('previous_campaign_outcome')['subscription_encoded']

    .mean()

    .reset_index()

)
plt.figure(figsize=(10, 6))

sns.barplot(
    
    x='previous_campaign_outcome',

    y='subscription_encoded',

    data=subscription_rate
)
plt.title('Subscription Rate by Previous Campaign Outcome')

plt.xlabel('Previous Campaign Outcome')

plt.ylabel('Average Subscription Rate')

plt.show()

#### 📌 Statistical Finding

The Chi-Square Test of Independence produced a highly significant result (`p < 0.05`), indicating that previous campaign outcomes and future subscription behavior are statistically associated.

This finding suggests that customers with successful historical campaign interactions demonstrate meaningfully different conversion behavior compared to customers with unsuccessful or no prior engagement history.

The analysis strengthens the relationship continuity hypothesis and indicates that prior positive engagement with the bank may significantly influence future responsiveness toward term deposit campaigns.

From a business perspective, this insight highlights the strategic value of customer relationship history and suggests that previously successful customers represent a high-potential segment for future campaign targeting and retention-focused outreach strategies.


### 13C.1 Previous Success Conversion Uplift Analysis

This analysis investigates how previous campaign outcomes influence future customer conversion probability.

The objective is to quantify how much more likely previously successful customers are to subscribe compared to customers with no prior successful engagement history.

This helps evaluate the strategic value of relationship continuity and historical customer trust.


In [ ]:
# Calculate conversion rate by previous campaign outcome

previous_success_analysis = (
    df_clean

    .groupby('previous_campaign_outcome')['subscription_encoded']

    .mean()

    .reset_index()

)

# Convert to percentage

previous_success_analysis['subscription_encoded'] = (
 
    previous_success_analysis['subscription_encoded'] * 100
)

# Display results

print(previous_success_analysis)

In [ ]:
# Plot conversion rate by previous campaign outcome

plt.figure(figsize=(10, 6))

sns.barplot(

    data=previous_success_analysis,

    x='previous_campaign_outcome',

    y='subscription_encoded'
)

# Chart formatting

plt.title('Conversion Rate by Previous Rate Outcome')

plt.xlabel('Previous Campaign Outcome')

plt.ylabel('Subscription Rate(%)')

plt.show()

In [ ]:
# Calculate Uplift

# Extract conversion rates

success_rate = previous_success_analysis[

    previous_success_analysis['previous_campaign_outcome'] == 'success'

]['subscription_encoded'].values[0]

new_customer_rate = previous_success_analysis[

    previous_success_analysis['previous_campaign_outcome'] == 'nonexistent'

]['subscription_encoded'].values[0]

# Calculate uplift ratio

uplift = success_rate / new_customer_rate

print(f"Conversion Uplift: {uplift:.2f}x")

#### 📌 Business Finding

The uplift analysis reveals a substantial relationship continuity effect within the campaign data.

Customers with a previously successful campaign interaction are approximately **7.37 times more likely** to subscribe compared to customers with no prior successful engagement history.

This represents one of the strongest behavioral signals identified in the project and highlights the strategic importance of customer trust, historical satisfaction, and relationship continuity in banking campaign performance.

From a business perspective, this finding suggests that previously successful customers should be treated as a high-priority segment for future targeting, retention-focused campaigns, and personalized banking outreach strategies.


# STEP 14 — SQL Analytical Environment Setup

## Objective

Move the finalized analytical dataset into SQL Server to enable scalable business querying, KPI generation, and downstream BI development.

At this stage, the project transitions from exploratory analysis into structured analytical engineering.

The SQL environment will serve as the central layer for:

* campaign performance analysis
* customer segmentation querying
* conversion KPI generation
* operational reporting
* Power BI integration

The dataset being imported is no longer raw operational data. It already contains:

* cleaned structures
* standardized business naming
* engineered analytical features
* segmentation dimensions
* statistical enrichment fields

This creates a business-ready analytical table optimized for enterprise reporting and decision-support workflows.


## 14A. Export Analytical Dataset

Export the finalized analytical dataset from Python for SQL Server integration and downstream BI analysis.


In [ ]:
# Export final analytical dataset for SQL Server integration

df_clean.to_csv(

    'banking_analytical_dataset.csv',

    index=False
)
print("Analytical dataset exported successfully.")